In [ ]:
import warnings
import pandas as pd
import mario

from common import load_config, db_path, should_skip

from database_properties import properties

# Keep notebook output readable while preserving MARIO progress messages.
mario.set_log_verbosity('info')
warnings.filterwarnings('ignore')

# How to react when a target CSV already exists in export/:
#   'ask'       -> prompt y/N for each file
#   'skip'      -> always reuse the existing export
#   'overwrite' -> always recompute and replace the export
OVERWRITE_POLICY = 'overwrite'

# Load path templates from paths.yml. `base` is only populated when using the
# older shared-root configuration; with the current explicit-path layout it is None.
cfg, base = load_config()

## EXIOBASE monetary IOT

In [ ]:
p = properties['exiobase_monetary_iot']
name, table = p['name'], p['table']

for version in p['versions']:
    for year in range(1998,2024):#p['years']:
        for system in p['systems']:
            
            # EXIOBASE archives are resolved from the configured path template.
            archive = f'IOT_{year}_{system}'
            folder = db_path(
                cfg, base, 'EXIOBASE',
                version=version, table=table, system=system, year=year, archive=archive,
            )
            print(f'\n=== {name} {version} {table} {system} {year} ===')

            # Respect the overwrite policy before doing any download or parsing work.
            if should_skip(name, table, version, year, system, policy=OVERWRITE_POLICY):
                continue

            # Parse the database, aggregate GHGs, then keep only electricity sectors.
            try:
                db = mario.parse_exiobase(
                    path=str(folder),
                    table=table,
                    unit='Monetary',
                )
            except Exception as exc:
                print(f'[skip] parse failed: {type(exc).__name__}: {exc}')
                continue

            db_aggr_reg = db.aggregate(
                io = {"Sector": pd.DataFrame({"Aggregation": ["All"] * len(db.sectors)},index=db.sectors)},
                ignore_nan=True, levels='Sector', inplace=False, zero_output_epsilon=None
                )
            db_aggr_sec = db.aggregate(
                io = {"Region": pd.DataFrame({"Aggregation": ["All"] * len(db.regions)},index=db.regions)},
                ignore_nan=True, levels='Region', inplace=False, zero_output_epsilon=None
                )
            
            db_aggr_reg.to_parquet(f"databases/{name}_{version}_{table}_{system}_{year}_region")
            db_aggr_sec.to_parquet(f"databases/{name}_{version}_{table}_{system}_{year}_sector")

## FIGARO

In [ ]:
p = properties['figaro']
name, table = p['name'], p['table']

for version in p['versions']:
    for year in range(2019,2022):#p['years']:
        # FIGARO is addressed through the configured full path template.
        path = db_path(cfg, base, 'FIGARO', table=table, year=year)
        print(f'\n=== {name} {version} {table} {year} ===')

        if should_skip(name, table, version, year, policy=OVERWRITE_POLICY):
            continue

        try:
            db = mario.parse_figaro(path=str(path), table=table, year=year)
        except Exception as exc:
            print(f'[skip] parse failed: {type(exc).__name__}: {exc}')
            continue

        db_aggr_reg = db.aggregate(
            io = {"Sector": pd.DataFrame({"Aggregation": ["All"] * len(db.sectors)},index=db.sectors)},
            ignore_nan=True, levels='Sector', inplace=False, zero_output_epsilon=None
            )
        db_aggr_sec = db.aggregate(
            io = {"Region": pd.DataFrame({"Aggregation": ["All"] * len(db.regions)},index=db.regions)},
            ignore_nan=True, levels='Region', inplace=False, zero_output_epsilon=None
            )
        
        db_aggr_reg.to_parquet(f"databases/{name}_{version}_{table}_{year}_region")
        db_aggr_sec.to_parquet(f"databases/{name}_{version}_{table}_{year}_sector")

## EORA26

In [ ]:
p = properties['eora26']
name, table = p['name'], p['table']

for version in p['versions']:
    for year in p['years']:
        # EORA26 is addressed through the configured full path template.
        path = db_path(cfg, base, 'EORA26', year=year)
        print(f'\n=== {name} {version} {table} {year} ===')

        if should_skip(name, table, version, year, policy=OVERWRITE_POLICY):
            continue

        try:
            db = mario.parse_eora(path=str(path), multi_region=True)
        except Exception as exc:
            print(f'[skip] parse failed: {type(exc).__name__}: {exc}')
            continue

        db_aggr_reg = db.aggregate(
            io = {"Sector": pd.DataFrame({"Aggregation": ["All"] * len(db.sectors)},index=db.sectors)},
            ignore_nan=True, levels='Sector', inplace=False, zero_output_epsilon=None
            )
        db_aggr_sec = db.aggregate(
            io = {"Region": pd.DataFrame({"Aggregation": ["All"] * len(db.regions)},index=db.regions)},
            ignore_nan=True, levels='Region', inplace=False, zero_output_epsilon=None
            )
        
        db_aggr_reg.to_parquet(f"databases/{name}_{version}_{table}_{year}_region")
        db_aggr_sec.to_parquet(f"databases/{name}_{version}_{table}_{year}_sector")

## OECD ICIO

In [ ]:
p = properties['oecd-icio']
name, table = p['name'], p['table']

for version in p['versions']:
    for year in [2019]:#p['years']:
        # OECD-ICIO is addressed through the configured full path template.
        path = db_path(cfg, base, 'OECD-ICIO', year=year)
        print(f'\n=== {name} {version} {table} {year} ===')

        if should_skip(name, table, version, year, policy=OVERWRITE_POLICY):
            continue

        try:
            db = mario.parse_oecd(path=str(path))
        except Exception as exc:
            print(f'[skip] parse failed: {type(exc).__name__}: {exc}')
            continue

        db_aggr_reg = db.aggregate(
            io = {"Sector": pd.DataFrame({"Aggregation": ["All"] * len(db.sectors)},index=db.sectors)},
            ignore_nan=True, levels='Sector', inplace=False, zero_output_epsilon=None
            )
        db_aggr_sec = db.aggregate(
            io = {"Region": pd.DataFrame({"Aggregation": ["All"] * len(db.regions)},index=db.regions)},
            ignore_nan=True, levels='Region', inplace=False, zero_output_epsilon=None
            )
        
        db_aggr_reg.to_parquet(f"databases/{name}_{version}_{table}_{year}_region")
        db_aggr_sec.to_parquet(f"databases/{name}_{version}_{table}_{year}_sector")

## EMERGING

In [ ]:
p = properties['emerging']
name, table = p['name'], p['table']

for version in p['versions']:

    for year in [2015]:#p['years']:
        path = db_path(cfg, base, f'EMERGING', year=year)
        print(f'\n=== {name} {version} {table} {year} ===')

        if should_skip(name, table, version, year, policy=OVERWRITE_POLICY):
            continue

        try:
            db = mario.parse_emerging(path=str(path), year=year)
        except Exception as exc:
            print(f'[skip] parse failed: {type(exc).__name__}: {exc}')
            continue

        db_aggr_reg = db.aggregate(
            io = {"Sector": pd.DataFrame({"Aggregation": ["All"] * len(db.sectors)},index=db.sectors)},
            ignore_nan=True, levels='Sector', inplace=False, zero_output_epsilon=None
            )
        db_aggr_sec = db.aggregate(
            io = {"Region": pd.DataFrame({"Aggregation": ["All"] * len(db.regions)},index=db.regions)},
            ignore_nan=True, levels='Region', inplace=False, zero_output_epsilon=None
            )
        
        db_aggr_reg.to_parquet(f"databases/{name}_{version}_{table}_{year}_region")
        db_aggr_sec.to_parquet(f"databases/{name}_{version}_{table}_{year}_sector")

## ADB

In [ ]:
p = properties['adb']
name, table = p['name'], p['table']

for version in p['versions']:

    for year in p['years']:
        path = db_path(cfg, base, f'ADB', version=version)
        print(f'\n=== {name} {version} {table} {year} ===')

        if should_skip(name, table, version, year, policy=OVERWRITE_POLICY):
            continue

        try:
            db = mario.parse_adb(path=str(path), year=year)
        except Exception as exc:
            print(f'[skip] parse failed: {type(exc).__name__}: {exc}')
            continue

        db_aggr_reg = db.aggregate(
            io = {"Sector": pd.DataFrame({"Aggregation": ["All"] * len(db.sectors)},index=db.sectors)},
            ignore_nan=True, levels='Sector', inplace=False, zero_output_epsilon=None
            )
        db_aggr_sec = db.aggregate(
            io = {"Region": pd.DataFrame({"Aggregation": ["All"] * len(db.regions)},index=db.regions)},
            ignore_nan=True, levels='Region', inplace=False, zero_output_epsilon=None
            )
        
        db_aggr_reg.to_parquet(f"databases/{name}_{version}_{table}_{year}_region")
        db_aggr_sec.to_parquet(f"databases/{name}_{version}_{table}_{year}_sector")

## GLORIA

In [ ]:
p = properties['gloria']
name, table = p['name'], p['table']

for version in p['versions']:
    for year in [2012,2021,2022,2024]:#p['years']:
        path = db_path(cfg, base, 'GLORIA', year=year)
        print(f'\n=== {name} {version} {table} {year} ===')

        if should_skip(name, table, version, year, policy=OVERWRITE_POLICY):
            continue

        try:
            db = mario.parse_gloria(path=str(path), year=year)
        except Exception as exc:
            print(f'[skip] parse failed: {type(exc).__name__}: {exc}')
            continue

        db_aggr_reg = db.aggregate(
            io = {
                "Activity": pd.DataFrame({"Aggregation": ["All"] * len(db.activities)},index=db.activities),
                "Commodity": pd.DataFrame({"Aggregation": ["All"] * len(db.commodities)},index=db.commodities),
                },
            ignore_nan=True, levels=['Activity','Commodity'], inplace=False, zero_output_epsilon=None
            )
        db_aggr_sec = db.aggregate(
            io = {
                "Region": pd.DataFrame({"Aggregation": ["All"] * len(db.regions)},index=db.regions),
                },
            ignore_nan=True, levels='Region', inplace=False, zero_output_epsilon=None
            )
        
        db_aggr_reg.to_parquet(f"databases/{name}_{version}_{table}_{year}_region")
        db_aggr_sec.to_parquet(f"databases/{name}_{version}_{table}_{year}_sector")

## Regional trade totals

This section reloads every `*_region` parquet database, computes total inter-regional trade with `db.calc_trades()`, and exports one combined CSV with dataset metadata.

In [1]:
import pandas as pd
import mario
import os

mario.set_log_verbosity('critical')

all_trades = pd.DataFrame()
all_GDP = pd.DataFrame()

databases = sorted(os.listdir('databases'))
if '.DS_Store' in databases:
    databases.remove('.DS_Store') # --- IGNORE ---

for database in databases:
    name = database.split('_')[0]
    version = database.split('_')[1]
    table = database.split('_')[2]

    if database.endswith('region'):# and name != 'GLORIA':    
        if name == 'EXIOBASE' and version == '3.10.2':
            system = database.split('_')[3]
            year = database.split('_')[4]
            db = mario.parse_from_parquet(path=f"databases/{database}/flows",table=table,mode="flows")

        if name in ['EORA26', 'FIGARO', 'EMERGING', 'ADB', 'OECD-ICIO', 'GLORIA']:
            year = database.split('_')[3]
            db = mario.parse_from_parquet(path=f"databases/{database}/flows",table=table,mode="flows")

        trades = db.calc_trades()
        trades.index.names = ["Origin"]
        trades.columns.names = ["Destination"]
        trades = trades.stack().to_frame()
        trades.columns = ["Value"]

        if table == 'IOT':
            trades['Unit'] = db.units['Sector'].values[0,0]
        else:
            trades['Unit'] = db.units['Commodity'].values[0,0]

        trades['Name'] = name
        trades['Version'] = version
        trades['Table'] = table
        trades['Year'] = year
        try:
            trades['System'] = system
        except:
            trades['System'] = ''

        trades.reset_index(inplace=True)
        all_trades = pd.concat([all_trades, trades],axis=0)

        GDP = db.GDP()
        GDP.index.names = ["Region"]
        GDP.columns = ['Value']
        if table == 'IOT':
            GDP['Unit'] = db.units['Sector'].values[0,0]
        else:
            GDP['Unit'] = db.units['Commodity'].values[0,0]
        GDP['Name'] = name
        GDP['Version'] = version
        GDP['Table'] = table
        GDP['Year'] = year
        try:
            GDP['System'] = system
        except:
            GDP['System'] = ''

        GDP.reset_index(inplace=True)
        all_GDP = pd.concat([all_GDP, GDP], axis=0)

all_trades = all_trades[["Name", "Version", "Table", "System", "Year", "Origin", "Destination", "Unit", "Value"]]
all_trades.to_csv("results/trades.csv", index=False)

all_GDP = all_GDP[["Name", "Version", "Table", "System", "Year", "Region", "Unit", "Value"]]
all_GDP.to_csv("results/GDP.csv", index=False)

## Cross-database trade comparison (ISO3 + currency harmonization)

This section harmonizes country labels to ISO3 with `country_converter`, removes RoW-like regions, converts EUR/USD to a common currency, and builds comparison plots by database (`Name + Version + System`).

In [2]:
from pathlib import Path
import importlib
import plots
importlib.reload(plots)
from plots import (
    prepare_trade_comparison_dataframe,
    get_yearly_eur_usd_rates,
    export_trade_dashboard_html,
)
import pandas as pd

trades_raw = pd.read_csv(Path('results') / 'trades.csv')

# Parameters for harmonization/comparison
TARGET_CURRENCY = 'USD'  # 'USD' or 'EUR'
PLOT_YEAR = 2021  # set to None to aggregate across all years

years = sorted(pd.to_numeric(trades_raw['Year'], errors='coerce').dropna().astype(int).unique())
# Uses a year-dependent EUR->USD table (offline by default).
# Set source='forex' to try live retrieval and fallback to builtin when unavailable.
eur_usd_by_year = get_yearly_eur_usd_rates(years, source='builtin')

trades_cmp = prepare_trade_comparison_dataframe(
    trades_raw,
    target_currency=TARGET_CURRENCY,
    eur_usd_by_year=eur_usd_by_year
)
trades_cmp.to_csv("support/trades_comparison.csv", index=False)

/var/folders/7_/vg3_ld6n2pd73xyzt9dcdhlc0000gn/T/ipykernel_85320/812960282.py:12: DtypeWarning: Columns (0: Version, 1: System) have mixed types. Specify dtype option on import or set low_memory=False.
  trades_raw = pd.read_csv(Path('results') / 'trades.csv')


In [ ]:
# # Export a standalone HTML dashboard with Origin/Destination/Year dropdown menus.
# OUTPUT_HTML = 'plots/trade_dashboard.html'
# YEARS_FOR_DASHBOARD = range(2000, 2021)  # reduce size: pick only years you need
# TOP_N_OD_PAIRS = 3000             # keep only the most relevant OD pairs
# MIN_ABS_VALUE = 1.0              # drop tiny flows to shrink payload

# html_path = export_trade_dashboard_html(
#     pd.read_csv("support/trades_comparison.csv"),
#     output_path=OUTPUT_HTML,
#     years=YEARS_FOR_DASHBOARD,
#     top_n_pairs=TOP_N_OD_PAIRS,
#     min_abs_value=MIN_ABS_VALUE,
#     round_decimals=2,
# )
# print(f'Saved: {html_path}')

In [9]:
from pathlib import Path
import importlib
import plots
importlib.reload(plots)
from plots import (
    export_trade_dashboard_html,
)
import pandas as pd
import country_converter as coco

# AGGREGATION = 'G7'  # e.g. 'G20', 'EU', 'OECD', 'G7', ...

# cc = coco.CountryConverter()
# col = cc.data[AGGREGATION]
# regions = cc.data.loc[col.notna(), 'ISO3'].dropna().tolist()

regions = ['CAN','CHN','DEU','FRA','ITA', 'USA'] # --- IGNORE ---

html_path = export_trade_dashboard_html(
    pd.read_csv("support/trades_comparison.csv"),
    output_path="plots/trade_dashboard.html",
    regions=regions,
    years=[2015, 2018, 2021, 2023],#range(2019, 2025),
    min_abs_value=1.0,
    round_decimals=2,
    title=f'Trades and domestic consumption accounts - % variation from annual mean across MRIO databases',
)

/var/folders/7_/vg3_ld6n2pd73xyzt9dcdhlc0000gn/T/ipykernel_85320/3742675803.py:20: DtypeWarning: Columns (0: Version, 1: System) have mixed types. Specify dtype option on import or set low_memory=False.
  pd.read_csv("support/trades_comparison.csv"),


In [10]:
from plots import export_gdp_region_grid_html
import pandas as pd
import country_converter as coco

# AGGREGATION = 'G20'  # e.g. 'G20', 'EU', 'OECD', 'G7', ...

# cc = coco.CountryConverter()
# col = cc.data[AGGREGATION]
# regions = cc.data.loc[col.notna(), 'ISO3'].dropna().tolist()

regions = [
    'ARG', 'AUS', 'BRA', 'CAN', 'CHN', 'FRA', 'DEU', 'ESP', 'IND', 'IDN',
    'ITA', 'JPN', 'MEX', 'RUS', 'SAU', 'ZAF', 'KOR', 'TUR', 'GBR', 'USA'
]

html_path = export_gdp_region_grid_html(
    pd.read_csv("results/GDP.csv"),
    regions=regions,
    output_path="plots/gdp_region_grid.html",
    years=[2015, 2018, 2021, 2023],#range(2019, 2025),
    n_rows=4,
    n_cols=5,
    title="GDP comparison across databases - % variation from annual mean across MRIO databases",
)